# 10: A3max, a POST-HOC sensitivity variant of A3 (Kaggle or Colab)
**Chosen after seeing test results; reported as a separate, labelled column; never used in retention, rankings or
the primary Holm families.** A3 v2 (notebook 09) stays the primary abliteration attack, unchanged.

Why: on SDT, RL, AM_rand1 and AM_all one candidate layer passed A3's KL <= 0.1 benign-drift filter but barely
reduced dev refusal, so A3 left them nearly intact. A3max is identical to A3 v2 (same calibration direction, depth
window, candidate layers and dev scores) except that the layer is the **lowest dev refusal of all candidates, with
no KL filter**, applied the same way to all 12 runs (BASE, 7 team runs, AM, 3 controls) on harmful_test and
mmlu_test: 24 jobs (`configs/gen_a3max_test.yaml`, `aftermarket.attacks.a3max`).

Jobs are claimed on the Hub before they start, so Kaggle and Colab can run this at the same time; finished jobs are
pulled and skipped, so a re-run resumes. The plan cell prints the estimate and, per run, the layer A3max should pick
(predicted from the A3 v2 manifests' logged dev scores). Judge afterwards with `notebooks/06_judge.ipynb`.

- **Kaggle**: GPU T4 x2, secret `HF_TOKEN` (write; Add-ons -> Secrets), then **Save & Run All (Commit)**.
  `MAX_HOURS = 3.0`: no job starts after 3 h of session time (estimate: ~1.1 h on two T4s).
- **Colab**: Runtime -> T4 GPU, secret `HF_TOKEN` (key icon, notebook access on), Run all. After a disconnect, Run all again.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bennytimz/aftermarket/blob/main/notebooks/10_a3max.ipynb)

In [ ]:
# Session clock, platform. Kaggle charges the whole session (setup included), so the clock starts here.
import os, sys, time, uuid
SESSION_START = time.time()
IN_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.isdir("/kaggle") and not IN_COLAB
MAX_HOURS = 3.0 if ON_KAGGLE else 10.0  # Kaggle: no job starts after 3 h; Colab costs no Kaggle quota
# one owner id per session for the Hub job locks
os.environ.setdefault("AFTERMARKET_SESSION", f"{'kaggle' if ON_KAGGLE else 'colab' if IN_COLAB else 'local'}-{uuid.uuid4().hex[:8]}")
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if IN_COLAB else "other", "| max hours:", MAX_HOURS,
      "| session:", os.environ["AFTERMARKET_SESSION"])

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import subprocess
REPO_URL = "https://github.com/bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping the image's torch build fixed.
#     Colab: A1 trains a QLoRA adapter, so bitsandbytes / peft / accelerate are pinned as in requirements.txt.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
if IN_COLAB:
    !pip install -q bitsandbytes==0.50.2 peft==0.21.2 accelerate==1.15.0 -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face: Colab secrets (google.colab.userdata) on Colab, Kaggle secrets otherwise.
# Never prompts, so it also works in background "Save & Run All" runs.
from huggingface_hub import login, whoami
HF_USER = None
try:
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")  # key icon -> HF_TOKEN, notebook access on
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")  # Add-ons -> Secrets
    login(token=token, add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info: every visible T4 gets one worker (Kaggle: 2, Colab: 1)
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
GPUS = ",".join(str(i) for i in range(max(torch.cuda.device_count(), 1)))
print("GPUS =", GPUS)

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import os, json, sys
os.environ["AFTERMARKET_DEFENSE_DIR"] = "/kaggle/temp/aftermarket_defense" if os.path.isdir("/kaggle") else "/tmp/aftermarket_defense"
os.environ["AFTERMARKET_ADAPTERS"] = "/kaggle/temp/aftermarket_adapters" if os.path.isdir("/kaggle") else "/tmp/aftermarket_adapters"
os.environ["AFTERMARKET_TMP"] = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
for k in ("AFTERMARKET_DEFENSE_DIR", "AFTERMARKET_ADAPTERS"):
    os.makedirs(os.environ[k], exist_ok=True)
sys.path.insert(0, "src")
print("max hours:", MAX_HOURS, "| gpus:", GPUS)

## 1. Plan: estimate + predicted A3max layer per run (no GPU work)

In [ ]:
!PYTHONPATH=src python -m aftermarket.attacks.a3max --plan-only --max-hours {MAX_HOURS} --session-start {SESSION_START}

## 2. Run (claims each job on the Hub; skips jobs done or running elsewhere)

In [ ]:
!PYTHONPATH=src python -m aftermarket.attacks.a3max --gpus {GPUS} --max-hours {MAX_HOURS} --session-start {SESSION_START}

## 3. Status (always runs; stats only, no completion text)

In [ ]:
!PYTHONPATH=src python -m aftermarket.attacks.a3max --summary-only --max-hours {MAX_HOURS} --session-start {SESSION_START}